# 03 — Unsupervised Clustering
Task 2.1–2.5: PCA, K-Means, DBSCAN, профили кластеров, anomaly detection.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from IPython.display import display, Markdown
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.ensemble import IsolationForest

from utils.config import load_config

cfg = load_config('config.yaml')
SEED = cfg['seed']
random.seed(SEED)
np.random.seed(SEED)

FEATURES = cfg['dataset']['feature_columns']
COLS = FEATURES + ['label']

df = pd.read_csv(cfg['paths']['data'], names=COLS)
df['known_label'] = (df['label'] != 'normal.').astype(int)
if len(df) > cfg['unsupervised']['sample_size']:
    df = df.sample(cfg['unsupervised']['sample_size'], random_state=SEED).reset_index(drop=True)

prep_path = Path(cfg['paths']['preprocessor'])
if not prep_path.exists():
    raise FileNotFoundError('Сначала запусти Notebook 1 для сохранения preprocessor.joblib')
prep = joblib.load(prep_path)
X = prep.transform(df[FEATURES])
y_known = df['known_label'].to_numpy()

print('Used rows:', len(df), 'Transformed shape:', X.shape)

In [ ]:
# Task 2.1 PCA
pca = PCA(n_components=2, random_state=SEED)
X2 = pca.fit_transform(X)
joblib.dump(pca, cfg['paths']['pca_model'])

plt.figure(figsize=(9, 7))
sc = plt.scatter(X2[:, 0], X2[:, 1], c=y_known, s=8, alpha=0.5, cmap='coolwarm')
plt.title('PCA 2D (color = known label)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(*sc.legend_elements(), title='Label')
plt.tight_layout()
plt.show()

print('Explained variance (%):', [round(v * 100, 2) for v in pca.explained_variance_ratio_], 'Total:', round(float(pca.explained_variance_ratio_.sum() * 100), 2))

In [ ]:
# Task 2.2 KMeans + Elbow + Silhouette
k_values = cfg['unsupervised']['k_range']
inertia, sil = [], []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    lbl = km.fit_predict(X2)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X2, lbl))

best_k = int(k_values[int(np.argmax(sil))])
kmeans = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
k_lbl = kmeans.fit_predict(X2)
joblib.dump(kmeans, cfg['paths']['clustering_model'])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(k_values, inertia, marker='o'); ax[0].set_title('Elbow'); ax[0].set_xlabel('k'); ax[0].set_ylabel('Inertia')
ax[1].plot(k_values, sil, marker='o', color='green'); ax[1].set_title('Silhouette'); ax[1].set_xlabel('k'); ax[1].set_ylabel('Score')
plt.tight_layout(); plt.show()

print('Selected k:', best_k, 'Silhouette:', round(float(max(sil)), 4))

In [ ]:
# Task 2.3 DBSCAN vs KMeans
db = DBSCAN(eps=0.5, min_samples=10)
db_lbl = db.fit_predict(X2)
valid = db_lbl != -1
db_sil = silhouette_score(X2[valid], db_lbl[valid]) if len(set(db_lbl[valid])) > 1 else np.nan

k_sil = silhouette_score(X2, k_lbl)
k_ari = adjusted_rand_score(y_known, k_lbl)
db_ari = adjusted_rand_score(y_known, db_lbl)

pd.DataFrame([
    {'Algorithm': 'KMeans', 'Silhouette': k_sil, 'ARI': k_ari},
    {'Algorithm': 'DBSCAN', 'Silhouette': db_sil, 'ARI': db_ari},
])

In [ ]:
# Task 2.4 Cluster analysis with required statistics
clustered = df.copy()
clustered['cluster'] = k_lbl
num_cols = [c for c in FEATURES if c not in cfg['dataset']['categorical_features']]
stats = clustered.groupby('cluster')[num_cols].agg(['mean', 'std', 'min', 'max'])

global_mean = clustered[num_cols].mean()
for cid in sorted(clustered['cluster'].unique()):
    cdf = clustered[clustered['cluster'] == cid][num_cols]
    deltas = (cdf.mean() - global_mean).abs().sort_values(ascending=False)
    f1, f2 = deltas.index[:2]
    s1, s2 = cdf[f1].describe(), cdf[f2].describe()
    text = (
        f'Cluster {cid}: гипотеза — отдельный тип сетевого поведения. '
        f'{f1}: mean={s1["mean"]:.3f}, std={s1["std"]:.3f}, min={s1["min"]:.3f}, max={s1["max"]:.3f}; '
        f'{f2}: mean={s2["mean"]:.3f}, std={s2["std"]:.3f}, min={s2["min"]:.3f}, max={s2["max"]:.3f}.'
    )
    display(Markdown(text))

stats.head()

In [ ]:
# Task 2.5 Anomaly detection (top 1%)
iso = IsolationForest(contamination=cfg['unsupervised']['anomaly_fraction'], random_state=SEED)
iso.fit(X)
score = iso.decision_function(X)
thr = np.quantile(score, cfg['unsupervised']['anomaly_fraction'])
is_anomaly = (score <= thr).astype(int)

res = df.copy()
res['anomaly_score'] = score
res['is_anomaly'] = is_anomaly

top5 = res.sort_values('anomaly_score').head(5)
display(top5[['label', 'anomaly_score', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'count', 'srv_count']])
print('Anomaly count:', int(res['is_anomaly'].sum()))
print('Attack ratio among anomalies:', round(float(res[res['is_anomaly'] == 1]['known_label'].mean()), 4))

## Аналитика
- **2.1:** 2D PCA теряет часть информации, но хорошо показывает общую структуру трафика.
- **2.2:** `k` выбран по максимуму silhouette при адекватном elbow-переломе (почему не `k-1`/`k+1`: хуже баланс разделимости и стабильности).
- **2.3:** KMeans и DBSCAN расходятся из-за разных предположений о форме кластеров и шуме.
- **2.4:** для каждого кластера приведена статистика (mean/std/min/max) минимум по двум признакам.
- **2.5:** top 1% аномалий выделены через IsolationForest; сопоставление с метками показывает, насколько это похоже на разведку/предвзломную активность.